The synthetic Big 5 data was generated based on this paper:

Weisberg, Y. J., DeYoung, C. G., & Hirsh, J. B. (2011). Gender differences in personality across the ten aspects of the Big Five. Frontiers in psychology, 2, 11757.

In [1]:
import numpy as np
import pandas as pd
import json

In [21]:
def create_word_list(n_words, rng):
    WORD_POOL = [
        "apple", "river", "chair", "mountain", "window",
        "pencil", "forest", "bottle", "garden", "clock",
        "bridge", "orange", "blanket", "hammer", "candle",
        "ladder", "ocean", "basket", "camera", "feather",
        "guitar", "rabbit", "island", "mirror", "rocket",
        "cookie", "helmet", "lantern", "flower", "suitcase",
        "butterfly", "castle", "newspaper", "violin", "backpack",
        "turtle", "snowflake", "diamond", "rainbow", "airplane",
        "anchor", "balloon", "beach", "cactus", "dolphin",
        "eagle", "factory", "glove", "horizon", "jacket",
        "key", "library", "magnet", "notebook", "owl",
        "pillow", "quilt", "rope", "sandal", "teapot",
        "umbrella", "volcano", "whistle", "yacht", "zebra",
        "acorn", "beehive", "compass", "drum", "engine",
        "fountain", "garage", "harvest", "igloo", "jungle",
        "kettle", "lighthouse", "meadow", "nest", "orchard"
    ]
    words = rng.choice(WORD_POOL, size=n_words, replace=False)

    words_position_map = {
        word: position + 1
        for position, word in enumerate(words)
    }
    return words_position_map

In [22]:
def generate_personality_data(n: int, p_male: float = 0.5, seed=1) -> pd.DataFrame:
    """
    Generates a synthetic dataset of personality traits based on the Big Five model and their subfacets, with gender-specific correlations    
    """
    rng = np.random.default_rng(seed)
    facets = [
        "Enthusiasm", "Assertiveness",
        "Compassion", "Politeness",
        "Industriousness", "Orderliness",
        "Volatility", "Withdrawal",
        "Intellect", "Openness"
    ]

    corr = np.array([
        # Enth. Asse.  Comp.  Poli.  Ind.   Order.  Volat. With.  Inte.  Open.
        [1.00,  0.45,  0.46,  0.10,  0.21,  0.00,  -0.18, -0.35,  0.17,  0.15],  # Enth.
        [0.51,  1.00,  0.22,  -0.20, 0.34,  0.06,  -0.09, -0.40,  0.47,  0.20],  # Asse.
        [0.45,  0.25,  1.00,  0.38,  0.14,  0.03,  -0.17, -0.12,  0.24,  0.43],  # Comp.
        [0.12,  -0.17, 0.47,  1.00,  0.29,  0.23,  -0.35, -0.20,  -0.01, 0.07],  # Poli
        [0.22,  0.36,  0.21,  0.28,  1.00,  0.41,  -0.24, -0.51,  0.33,  -0.02], # Ind.
        [0.04,  0.08,  0.14,  0.21,  0.44,  1.00,  0.03,  -0.03, -0.02,  -0.07], # Order.
        [-0.15, -0.07, -0.15, -0.40, -0.36, -0.02, 1.00,  0.61,  -0.23,  0.00],  # Volat.
        [-0.37, -0.42, -0.14, -0.17, -0.54, -0.03, 0.62,  1.00,  -0.35,  0.03],  # With.
        [0.22,  0.48,  0.31,  0.10,  0.42,  0.04,  -0.21, -0.35, 1.00,   0.38],  # Inte.
        [0.13,  0.17,  0.39,  0.17,  0.01,  -0.03, 0.01,  0.04,  0.42,   1.00]   # Open.
    ])

    # This splits the correlation matrix into the upper an lower triangle (above for male, below for women)
    corr_male = corr.copy()
    corr_female = corr.copy()
    for i in range(len(facets)):
        for j in range(len(facets)):
            if i > j:
                corr_male[i, j] = corr[j, i]     # mirror upper triangle
            if i < j:
                corr_female[i, j] = corr[j, i]   # mirror lower triangle

    n_m = int(n * p_male)
    n_f = n - n_m

    Z_m = rng.multivariate_normal(
        mean=np.zeros(len(facets)),
        cov=corr_male,
        size=n_m
    )

    Z_f = rng.multivariate_normal(
        mean=np.zeros(len(facets)),
        cov=corr_female,
        size=n_f
    )

    df_m = pd.DataFrame(Z_m, columns=facets)
    df_m["Gender"] = "male"

    df_f = pd.DataFrame(Z_f, columns=facets)
    df_f["Gender"] = "female"

    df = pd.concat([df_m, df_f], ignore_index=True)
    params = {
        "Enthusiasm":     {"male": (3.40, 0.66), "female": (3.56, 0.68)},
        "Assertiveness":  {"male": (3.34, 0.64), "female": (3.28, 0.68)},
        "Compassion":     {"male": (3.78, 0.60), "female": (4.04, 0.56)},
        "Politeness":     {"male": (3.52, 0.61), "female": (3.74, 0.61)},
        "Industriousness":{"male": (3.25, 0.68), "female": (3.21, 0.73)},
        "Orderliness":    {"male": (3.38, 0.61), "female": (3.49, 0.63)},
        "Volatility":     {"male": (2.63, 0.75), "female": (2.86, 0.77)},
        "Withdrawal":     {"male": (2.73, 0.69), "female": (3.02, 0.71)},
        "Intellect":      {"male": (3.62, 0.61), "female": (3.48, 0.63)},
        "Openness":       {"male": (3.57, 0.61), "female": (3.74, 0.60)}
    }

    for facet, gender_params in params.items():
        for gender, (mu, sd) in gender_params.items():
            idx = df["Gender"] == gender
            z = (df.loc[idx, facet] - df.loc[idx, facet].mean()) / df.loc[idx, facet].std()
            df.loc[idx, facet] = mu + sd * z

    # clip values between 1-5 (This is a common range for Big 5 Tests)        
    for facet in facets:
        df[facet] = df[facet].clip(lower=1, upper=5)

    df["Extraversion"] = df[["Enthusiasm", "Assertiveness"]].mean(axis=1)
    df["Agreeableness"] = df[["Compassion", "Politeness"]].mean(axis=1)
    df["Conscientiousness"] = df[["Industriousness", "Orderliness"]].mean(axis=1)
    df["Neuroticism"] = df[["Volatility", "Withdrawal"]].mean(axis=1)
    df["Openness_Intellect"] = df[["Intellect", "Openness"]].mean(axis=1)

    # Processing
    df = df.round(2)
    df.insert(0, "Gender", df.pop("Gender"))
    df["raw_persona"] = df.apply(
        lambda row: "\n".join(f"{col}: {row[col]}" for col in df.columns if col not in ["llm_persona", "persona_generation_date"]),
        axis=1
    )
    # Assignemnt to split into random groups (for Kahnmemann 1 = AB order, 2 = BA order; and Gamblers Fallacy 1 = red, 2 = black)
    df["experiment_group"] = rng.choice([1, 2], size=len(df))
    for word_count in [20, 40, 80]:
        df[f"serial_position_effect_map_{word_count}"] = [
            json.dumps(create_word_list(word_count, rng))
            for _ in range(len(df))
        ]
    return df

In [31]:
df = generate_personality_data(n=300, p_male=0.5, seed=2)
df = df.head(2)
df.to_csv("Synthetic_Big5_data.csv", index=False)

# Tests:

In [28]:
df = df.drop(columns=["raw_persona", "serial_position_effect_map_20", "serial_position_effect_map_40", "serial_position_effect_map_80", "experiment_group"])

In [13]:
df.describe().round(2)

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect,experiment_group
count,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.00,300.0
mean,3.48,3.31,3.90,3.63,3.23,3.43,2.75,2.88,3.55,3.65,3.39,3.77,3.33,2.81,3.60,1.5
std,0.66,0.66,0.58,0.61,0.70,0.61,0.76,0.71,0.62,0.60,0.56,0.51,0.56,0.66,0.52,0.5
min,1.33,1.15,1.96,1.89,1.14,1.41,1.00,1.00,1.47,1.61,1.56,2.32,1.75,1.10,1.92,1.0
25%,3.06,2.90,3.53,3.20,2.79,3.00,2.25,2.38,3.09,3.24,3.06,3.38,2.97,2.36,3.28,1.0
50%,3.48,3.32,3.92,3.58,3.26,3.42,2.74,2.88,3.58,3.62,3.41,3.76,3.38,2.80,3.62,1.0
75%,3.88,3.70,4.30,4.08,3.65,3.86,3.24,3.38,3.99,4.09,3.74,4.11,3.70,3.26,3.98,2.0
max,5.00,5.00,5.00,5.00,5.00,5.00,4.99,4.90,5.00,5.00,5.00,4.91,5.00,4.61,4.95,2.0


In [14]:
df[df["Gender"] == "male"].describe().loc[['mean', 'std']].round(2)

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect,experiment_group
mean,3.40,3.34,3.78,3.52,3.25,3.38,2.63,2.73,3.62,3.57,3.37,3.65,3.31,2.68,3.59,1.54
std,0.66,0.64,0.60,0.60,0.68,0.60,0.75,0.69,0.61,0.61,0.57,0.50,0.54,0.64,0.53,0.50


In [15]:
df[df["Gender"] == "female"].describe().loc[['mean', 'std']].round(2)

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect,experiment_group
mean,3.55,3.28,4.03,3.74,3.21,3.49,2.86,3.02,3.48,3.73,3.42,3.88,3.35,2.94,3.61,1.45
std,0.66,0.68,0.54,0.61,0.73,0.63,0.76,0.71,0.62,0.59,0.56,0.49,0.58,0.67,0.51,0.50


In [17]:
df[df["Gender"] == "male"].drop(columns="Gender").corr().round(2)

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect,experiment_group
Enthusiasm,1.00,0.52,0.56,0.12,0.22,-0.01,-0.28,-0.47,0.28,0.29,0.88,0.41,0.13,-0.42,0.33,0.03
Assertiveness,0.52,1.00,0.32,-0.21,0.43,0.07,-0.14,-0.54,0.51,0.38,0.87,0.06,0.31,-0.37,0.51,0.03
Compassion,0.56,0.32,1.00,0.39,0.17,0.08,-0.27,-0.26,0.28,0.44,0.50,0.83,0.15,-0.30,0.41,-0.14
Politeness,0.12,-0.21,0.39,1.00,0.17,0.25,-0.31,-0.14,0.03,-0.05,-0.05,0.83,0.25,-0.26,-0.01,-0.07
Industriousness,0.22,0.43,0.17,0.17,1.00,0.41,-0.17,-0.51,0.51,0.11,0.37,0.20,0.86,-0.37,0.35,-0.01
Orderliness,-0.01,0.07,0.08,0.25,0.41,1.00,0.09,0.01,0.05,-0.08,0.04,0.20,0.82,0.06,-0.02,-0.01
Volatility,-0.28,-0.14,-0.27,-0.31,-0.17,0.09,1.00,0.56,-0.31,-0.11,-0.24,-0.35,-0.06,0.90,-0.24,0.05
Withdrawal,-0.47,-0.54,-0.26,-0.14,-0.51,0.01,0.56,1.00,-0.58,-0.21,-0.58,-0.24,-0.32,0.87,-0.45,0.07
Intellect,0.28,0.51,0.28,0.03,0.51,0.05,-0.31,-0.58,1.00,0.52,0.45,0.18,0.35,-0.50,0.87,-0.10
Openness,0.29,0.38,0.44,-0.05,0.11,-0.08,-0.11,-0.21,0.52,1.00,0.38,0.23,0.02,-0.18,0.87,-0.13


In [29]:
df[df["Gender"] == "female"].drop(columns="Gender").corr().round(2)

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
Enthusiasm,1.00,0.49,0.40,0.18,0.23,0.09,-0.10,-0.37,0.14,-0.01,0.86,0.32,0.20,-0.26,0.08
Assertiveness,0.49,1.00,0.18,-0.16,0.38,0.04,-0.02,-0.40,0.44,0.17,0.87,-0.00,0.26,-0.23,0.36
Compassion,0.40,0.18,1.00,0.56,0.28,0.20,-0.08,-0.06,0.28,0.26,0.33,0.87,0.28,-0.08,0.31
Politeness,0.18,-0.16,0.56,1.00,0.26,0.23,-0.39,-0.19,0.11,0.18,0.01,0.89,0.29,-0.33,0.16
Industriousness,0.23,0.38,0.28,0.26,1.00,0.47,-0.33,-0.52,0.42,0.13,0.36,0.31,0.88,-0.48,0.32
Orderliness,0.09,0.04,0.20,0.23,0.47,1.00,-0.01,-0.02,-0.03,-0.07,0.08,0.24,0.83,-0.02,-0.06
Volatility,-0.10,-0.02,-0.08,-0.39,-0.33,-0.01,1.00,0.57,-0.14,0.01,-0.07,-0.28,-0.21,0.89,-0.08
Withdrawal,-0.37,-0.40,-0.06,-0.19,-0.52,-0.02,0.57,1.00,-0.32,-0.06,-0.45,-0.14,-0.34,0.88,-0.23
Intellect,0.14,0.44,0.28,0.11,0.42,-0.03,-0.14,-0.32,1.00,0.52,0.34,0.22,0.25,-0.26,0.88
Openness,-0.01,0.17,0.26,0.18,0.13,-0.07,0.01,-0.06,0.52,1.00,0.10,0.25,0.04,-0.03,0.86
